# T14: Day 2 — Incremental Data Generation

## What You'll Learn

Day 1 is generating your initial dataset. **Day 2** is where it gets interesting:
new rows arrive, existing records change, some get deleted. This is the
**Change Data Capture (CDC)** pattern that drives every real-world data pipeline.

Spindle's incremental engine lets you generate realistic deltas on top of any
existing dataset — inserts, updates, and soft deletes — with full control over
rates, state transitions, and timestamps.

In this tutorial you will:

1. **Understand** incremental/CDC patterns and why they matter
2. **Generate** an initial baseline dataset
3. **Produce** incremental changes (inserts, updates, deletes)
4. **Inspect** delta tags (`_delta_type`, `_delta_timestamp`)
5. **Apply** state transitions (e.g., order pending -> shipped -> delivered)
6. **Create** time-travel snapshots spanning multiple months

## Why Incremental Generation?

| Static Dataset | Incremental Dataset |
|---|---|
| One snapshot in time | Evolves over time like production data |
| Cannot test CDC pipelines | Produces INSERT/UPDATE/DELETE deltas |
| No state transitions | Orders move from pending to shipped to delivered |
| No late-arriving data | Configurable late arrivals and out-of-order events |

## Prerequisites

- `sqllocks-spindle` installed
- Completed T01 (basic generation)

## Time Estimate

**~15 minutes**

In [1]:
# Uncomment the line below if sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

Installation cell ready. Uncomment the pip line above if needed.


## Step 1 — Generate the Baseline Dataset

**What we're about to do:** Generate an initial retail dataset. This is our
"Day 1" snapshot — the starting point that incremental changes will build upon.

**Why this matters:** Every CDC pipeline needs a baseline. In production, this
is your initial full load. The incremental engine needs this baseline to know
which records exist, what their current states are, and what key ranges are
already occupied.

**What to expect:** A standard retail dataset with customers, orders, products,
and order lines — our Day 1 snapshot.

In [2]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.domains.retail import RetailDomain

# Generate initial baseline
spindle = Spindle()
result = spindle.generate(domain=RetailDomain(), scale="small", seed=42)

print("Day 1 Baseline — Retail Domain")
print("=" * 45)
for table_name, df in result.tables.items():
    print(f"  {table_name:<20} {len(df):>8,} rows")

total = sum(len(df) for df in result.tables.values())
print(f"\nTotal rows: {total:,}")

# Show current order status distribution
if "order" in result.tables:
    orders = result.tables["order"]
    if "status" in orders.columns:
        print(f"\n=== Order Status Distribution (Day 1) ===")
        for status, count in orders["status"].value_counts().items():
            print(f"  {status:<15} {count:>6,}")

Day 1 Baseline — Retail Domain
  customer                1,000 rows
  address                 1,500 rows
  product_category           50 rows
  product                   500 rows
  promotion                 200 rows
  store                     150 rows
  order                   5,000 rows
  order_line             12,500 rows
  return                    850 rows

Total rows: 21,750

=== Order Status Distribution (Day 1) ===
  completed        3,862
  returned           433
  shipped            398
  cancelled          204
  processing         103


## Step 2 — Generate Incremental Changes

**What we're about to do:** Use the `ContinueEngine` to generate a batch of
incremental changes — new inserts, updates to existing records, and soft
deletes. The `ContinueConfig` controls exactly how many of each.

**Why this matters:** The continue engine is Spindle's CDC simulator. It
understands your schema's foreign keys, so new orders reference existing
customers, and updates modify realistic fields (not primary keys). The
`insert_count`, `update_fraction`, and `delete_fraction` parameters map
directly to the CDC patterns you'll see in production.

**What to expect:** A delta result containing tagged records — each marked
with `_delta_type` (insert, update, delete) and `_delta_timestamp`.

In [3]:
from sqllocks_spindle.incremental import ContinueEngine, ContinueConfig

engine = ContinueEngine()
config = ContinueConfig(
    insert_count=50,          # 50 new records across tables
    update_fraction=0.1,      # update 10% of existing records
    delete_fraction=0.02,     # soft-delete 2% of existing records
    state_transitions={
        "order.status": {
            "pending":  {"shipped": 0.7, "cancelled": 0.3},
            "shipped":  {"delivered": 0.9, "returned": 0.1},
        }
    },
    seed=42,
)

delta = engine.continue_from(result, config=config)
print(delta.summary())

Incremental Generation Result
  customer: +50 inserts, ~100 updates, -20 deletes
  address: +50 inserts, ~150 updates, -30 deletes
  product_category: +50 inserts, ~5 updates, -1 deletes
  product: +50 inserts, ~50 updates, -10 deletes
  promotion: +50 inserts, ~20 updates, -4 deletes
  store: +50 inserts, ~15 updates, -3 deletes
  order: +50 inserts, ~500 updates, -100 deletes
  order_line: +50 inserts, ~1250 updates, -250 deletes
  return: +50 inserts, ~85 updates, -17 deletes


C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\sqllocks_spindle\incremental\continue_engine.py:464: UserWarning: you are shuffling a 'ArrowStringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(subset)
C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\sqllocks_spindle\incremental\continue_engine.py:464: UserWarning: you are shuffling a 'ArrowStringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(subset)
C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\sqllocks_spindle\incremental\continue_engine.py:464: UserWarning: you are shuffling a 'ArrowStringArray' objec

## Step 3 — Inspect Delta Tags

**What we're about to do:** Examine the `_delta_type` and `_delta_timestamp`
columns that the continue engine adds to every changed record.

**Why this matters:** These metadata columns are exactly what CDC pipelines
consume. A Fabric Data Pipeline or Spark notebook can filter on `_delta_type`
to route inserts to append operations, updates to merge/upsert, and deletes
to soft-delete flags. The `_delta_timestamp` enables watermark-based
incremental loading.

**What to expect:** A breakdown of delta types by table, plus sample rows
showing the metadata columns.

In [4]:
# Inspect delta tags across all tables
print("=== Delta Breakdown by Table ===")
for table_name, df in delta.combined.items():
    if "_delta_type" in df.columns:
        counts = df["_delta_type"].value_counts()
        parts = [f"{dtype}: {count}" for dtype, count in counts.items()]
        print(f"  {table_name:<20} {len(df):>5} changes  |  {', '.join(parts)}")

# Show sample delta records
print("\n=== Sample Delta Records ===")
for table_name, df in delta.combined.items():
    if "_delta_type" in df.columns and len(df) > 0:
        print(f"\n--- {table_name} (first 3 changes) ---")
        display_cols = [c for c in df.columns if not c.startswith("_")] + ["_delta_type", "_delta_timestamp"]
        display_cols = [c for c in display_cols if c in df.columns]
        print(df[display_cols].head(3).to_string(index=False))
        break  # just show one table as a sample

=== Delta Breakdown by Table ===
  customer               170 changes  |  UPDATE: 100, INSERT: 50, DELETE: 20
  address                230 changes  |  UPDATE: 150, INSERT: 50, DELETE: 30
  product_category        56 changes  |  INSERT: 50, UPDATE: 5, DELETE: 1
  product                110 changes  |  INSERT: 50, UPDATE: 50, DELETE: 10
  promotion               74 changes  |  INSERT: 50, UPDATE: 20, DELETE: 4
  store                   68 changes  |  INSERT: 50, UPDATE: 15, DELETE: 3
  order                  650 changes  |  UPDATE: 500, DELETE: 100, INSERT: 50
  order_line            1550 changes  |  UPDATE: 1250, DELETE: 250, INSERT: 50
  return                 152 changes  |  UPDATE: 85, INSERT: 50, DELETE: 17

=== Sample Delta Records ===

--- customer (first 3 changes) ---
 customer_id first_name last_name                   email gender loyalty_tier                   signup_date is_active _delta_type           _delta_timestamp
        1001      Kelly Rodriguez georgecline@example.org

## Step 4 — State Transitions in Action

**What we're about to do:** Focus specifically on the order status transitions
we configured — `pending -> shipped/cancelled` and `shipped -> delivered/returned`.
We'll compare the status distribution before and after the incremental run.

**Why this matters:** State machines are everywhere in business data — order
statuses, claim statuses, ticket workflows, employee lifecycle stages. The
continue engine's `state_transitions` parameter lets you define transition
probabilities that mirror your real business process, producing deltas that
exercise your pipeline's merge logic realistically.

**What to expect:** Pending orders will have moved to shipped or cancelled;
shipped orders will have moved to delivered or returned.

In [5]:
# Compare order status before and after
if "order" in result.tables and "order" in delta.combined:
    before = result.tables["order"]
    after_changes = delta.combined["order"]

    print("=== Order Status — Before (Day 1) ===")
    if "status" in before.columns:
        for status, count in before["status"].value_counts().items():
            print(f"  {status:<15} {count:>6,}")

    print("\n=== Status Changes in Delta ===")
    updates = after_changes[after_changes["_delta_type"] == "update"] if "_delta_type" in after_changes.columns else after_changes
    if "status" in updates.columns and len(updates) > 0:
        print(f"  Updated orders: {len(updates)}")
        for status, count in updates["status"].value_counts().items():
            print(f"  -> {status:<15} {count:>6,}")
    else:
        print("  No status updates in this delta batch.")

    print("\nState transitions move orders through their lifecycle:")
    print("  pending  -> shipped (70%) / cancelled (30%)")
    print("  shipped  -> delivered (90%) / returned (10%)")

=== Order Status — Before (Day 1) ===
  completed        3,862
  returned           433
  shipped            398
  cancelled          204
  processing         103

=== Status Changes in Delta ===
  No status updates in this delta batch.

State transitions move orders through their lifecycle:
  pending  -> shipped (70%) / cancelled (30%)
  shipped  -> delivered (90%) / returned (10%)


## Step 5 — Time-Travel Snapshots

**What we're about to do:** Use the `TimeTravelEngine` to generate a complete
6-month history of a retail dataset — not just one delta, but a full sequence
of monthly snapshots showing how the data evolves over time.

**Why this matters:** Time-travel generation is the most powerful pattern for
testing data pipelines. It produces the complete history that your lakehouse
would accumulate over months of operation — including growth trends, churn,
and seasonal patterns. This lets you test time-based queries, SCD Type 2
implementations, and month-over-month dashboards without waiting months for
real data to accumulate.

**What to expect:** A time-travel result containing monthly snapshots with
growing data volumes, configured growth rate (5%) and churn rate (1%).

In [6]:
from sqllocks_spindle.incremental.time_travel import TimeTravelEngine, TimeTravelConfig

tt_engine = TimeTravelEngine()
tt_config = TimeTravelConfig(
    months=6,
    growth_rate=0.05,    # 5% month-over-month growth
    churn_rate=0.01,     # 1% monthly churn
    seed=42,
)

tt_result = tt_engine.generate(
    domain=RetailDomain(),
    config=tt_config,
    scale="small",
)

print(tt_result.summary())

# Explore month-over-month growth
if hasattr(tt_result, 'snapshots'):
    print("\n=== Monthly Snapshot Summary ===")
    print(f"{'Month':<12} {'Tables':>8} {'Total Rows':>12} {'vs Prior':>10}")
    print("-" * 45)

    prev_rows = 0
    for snapshot in tt_result.snapshots:
        total_rows = sum(len(df) for df in snapshot.tables.values())
        change = f"+{total_rows - prev_rows:,}" if prev_rows > 0 else "baseline"
        print(f"  Month {snapshot.month_index:<4} {len(snapshot.tables):>8} {total_rows:>12,} {change:>10}")
        prev_rows = total_rows

    print(f"\nNet growth over 6 months: {prev_rows - sum(len(df) for df in tt_result.snapshots[0].tables.values()):,} rows")
else:
    print("Time-travel result structure:")
    print(f"  Tables: {list(tt_result.tables.keys())}")
    print(f"  Total rows: {sum(len(df) for df in tt_result.tables.values()):,}")

Time-Travel Result
Domain: retail
Snapshots: 7

  Month           Date          Tables       Rows
  ----------------------------------------------
  Month 0        2023-01-01         9     21,750
  Month 1        2023-02-01         9     22,620
  Month 2        2023-03-01         9     23,526
  Month 3        2023-04-01         9     24,468
  Month 4        2023-05-01         9     25,448
  Month 5        2023-06-01         9     26,466
  Month 6        2023-07-01         9     27,525

=== Monthly Snapshot Summary ===
Month          Tables   Total Rows   vs Prior
---------------------------------------------
  Month 0           9       21,750   baseline
  Month 1           9       22,620       +870
  Month 2           9       23,526       +906
  Month 3           9       24,468       +942
  Month 4           9       25,448       +980
  Month 5           9       26,466     +1,018
  Month 6           9       27,525     +1,059

Net growth over 6 months: 5,775 rows


## What's Next?

You've mastered Spindle's incremental data generation — from single-batch CDC
deltas to multi-month time-travel histories. This is the foundation for testing
any data pipeline that handles change over time.

| Notebook | What You'll Learn |
|----------|-------------------|
| **F08: File-Drop Ingestion** | Simulate daily file drops with CDC deltas for pipeline testing |
| **F10: Month-End Close** | Use time-travel to simulate 12 months of financial data evolution |
| **T13: Composite Domains** | Combine multiple domains before applying incremental changes |
| **F01: Medallion Architecture** | Build a bronze/silver/gold lakehouse from incremental data |

**Key takeaways:**
- `ContinueEngine` generates CDC-style deltas (inserts, updates, deletes) from any baseline
- Every delta record is tagged with `_delta_type` and `_delta_timestamp` for pipeline consumption
- `state_transitions` model realistic business process flows (order lifecycle, claim adjudication, etc.)
- `TimeTravelEngine` produces multi-month snapshot histories with configurable growth and churn
- Deterministic seeds ensure reproducible deltas for regression testing